# Notebook 04: Forensic Signal Analysis - Green vs Red Zones

## Strategic Objective
Analyze the divergence between **Bullish Continuation (Green)** and **Market Exhaustion (Red)**. We identify the exact 'DNA' of the Top 10 shocks shown in the charts.

---

In [9]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import bigquery
from dotenv import load_dotenv

# THE NUCLEAR FIX: Always work from the project root
CURRENT_DIR = os.getcwd()
ROOT_DIR = os.path.abspath(os.path.join(CURRENT_DIR, '..')) if os.path.basename(CURRENT_DIR) == 'notebooks' else CURRENT_DIR
os.chdir(ROOT_DIR)

load_dotenv('.env')
sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = [15, 7]

project_id = os.getenv('GCP_PROJECT_ID')
dataset_id = os.getenv('GCP_DATASET_ID')
client = bigquery.Client(project=project_id)

query = f"SELECT * FROM `{project_id}.{dataset_id}.vw_looker_master_intelligence` ORDER BY ds ASC"
df = client.query(query).to_dataframe()
df['ds'] = pd.to_datetime(df['ds'])

if 'ml_volume_zscore' not in df.columns and 'zscore_30d' in df.columns:
    df['ml_volume_zscore'] = df['zscore_30d']

print(f"[SUCCESS] Forensic data loaded: {len(df)} records.")

/Users/charlie/ai_projects/portafolio/crypto-ml-predictor/venv/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


[SUCCESS] Forensic data loaded: 1970 records.


## 1. Classifying the battlefield
We isolate the signals and categorize them into Success (Green) and Failure (Red).

In [10]:
asset_id = 'bitcoin'
subset = df[df['id'] == asset_id].copy()
signals = subset[subset['ml_prediction'] == 1].copy()

def categorize_outcome(ret):
    if ret > 0.05: return 'GREEN (Momentum)'
    if ret < -0.05: return 'RED (Exhaustion)'
    return 'GRAY (Neutral)'

signals['outcome'] = signals['return_t30'].apply(categorize_outcome)
print("Outcome Distribution for Bitcoin Signals:")
print(signals['outcome'].value_counts())

Outcome Distribution for Bitcoin Signals:
outcome
GRAY (Neutral)      6
GREEN (Momentum)    5
RED (Exhaustion)    1
Name: count, dtype: int64


## 2. Forensic DNA Comparison
Comparing feature averages to find the trigger for exhaustion.

In [11]:
analysis_cols = ['ml_volume_zscore', 'ml_trend_ratio', 'ml_probability']
comparison = signals.groupby('outcome')[analysis_cols].mean()
display(comparison)

,ml_volume_zscore,ml_trend_ratio,ml_probability
outcome,,,
GRAY (Neutral),1.632510,1.067813,0.531178
GREEN (Momentum),-0.007356,0.945979,0.614809
RED (Exhaustion),2.186257,1.102581,0.579910


## 3. Tsunami DNA Table: The Top 20 Evidence
Isolating the exact metrics of the events shown in the impact charts.

In [ ]:
tsunami_top_10 = subset.sort_values('ml_volume_zscore', ascending=False).head(20).copy()
tsunami_top_10['outcome'] = tsunami_top_10['return_t30'].apply(categorize_outcome)

print("=== DNA OF THE TOP 10 SHOCKS ===")
forensic_cols = ['ds', 'ml_volume_zscore', 'ml_trend_ratio', 'ml_probability', 'outcome', 'return_t30']
display(tsunami_top_10[forensic_cols].sort_values('ds'))

=== DNA OF THE TOP 10 SHOCKS ===


,ds,ml_volume_zscore,ml_trend_ratio,ml_probability,outcome,return_t30
181,2025-05-22 00:00:00+00:00,2.186257,1.102581,0.579910,RED (Exhaustion),-0.058138
343,2025-06-23 00:00:00+00:00,2.264035,0.952746,0.553348,GREEN (Momentum),0.189417
346,2025-06-24 00:00:00+00:00,1.912580,0.997489,0.507234,GREEN (Momentum),0.124322
432,2025-07-11 00:00:00+00:00,2.287432,1.083632,0.356238,GRAY (Neutral),0.005440
438,2025-07-12 00:00:00+00:00,2.514922,1.096410,0.513304,GRAY (Neutral),0.014424
452,2025-07-15 00:00:00+00:00,2.591770,1.104172,0.505583,GRAY (Neutral),0.031104
458,2025-07-16 00:00:00+00:00,2.779252,1.080288,0.507671,GRAY (Neutral),0.006181
505,2025-07-26 00:00:00+00:00,2.531534,1.035352,0.367610,GRAY (Neutral),-0.035233
801,2025-09-23 00:00:00+00:00,2.365187,0.998740,0.355784,GRAY (Neutral),-0.045062
815,2025-09-26 00:00:00+00:00,2.433748,0.965976,0.163616,GRAY (Neutral),0.024382


## 4. Final Verdict: The Decision Guide

| Zone Type | DNA Characteristics | Action Recommendation |
| :--- | :--- | :--- |
| **GREEN (Continuation)** | Z-Score < 2.5 and Trend < 1.05 | **BUY.** Healthy institutional entry. |
| **RED (Exhaustion)** | Z-Score > 2.8 and Trend > 1.10 | **DANGER.** Buying climax at a peak. EXPECT CRASH. |
| **GRAY (Neutral)** | Probability < 55% | **WAIT.** Insufficient evidence. |

---

## 5. Auditor Final Verdict: The February 2026 Discovery
**Critical Insight:** During the forensic audit of the **February 6th, 2026** signal (Row 1480), we identified a 'False Bearish' prediction (Probability 29% but Green Outcome).

### Root Cause Analysis:
- The model currently penalizes high volume spikes based on the **October 2025 Exhaustion** pattern.
- However, February 2026 was a **Capitulation Bottom** (Trend Ratio 0.71). 
- **Learning:** Structural Volume Shocks are context-dependent. High volume at local tops leads to Red Zones, while high volume at deep bottoms leads to Green Zones.